# 28. The longest login streak

**Track:** sql | **Difficulty:** hard | **Tags:** gaps and islands, streaks, row_number, window functions, deduplication

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("028-longest-login-streak")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "app_logins": """
UEFSMRUEFbADFfIBTBU2FQASAADYAQQEAAkBAAsJBwQAEw0IAAYNCAARDQgAAQ0IAAcNCAAhDQgAAg0IAAkNCAAFDQgAAw0IAA0N
CAAUDQgADw0IABYNCAAKDQgAIw0IABANCAAIDQgADg0IABINCAAMDQgAIA0IABUNCDwiAAAAAAAAAB8AAAAAAAAAFQAViAUVkgUs
FeQHFRAVBhUGHBgIIwAAAAAAAAAYCAEAAAAAAAAAFgAoCCMAAAAAAAAAGAgBAAAAAAAAABERAAAAxAL0QwEDAAAA5AcBBX8giBFI
AcEclIACiSzDlmrItbdgfOnJN+GkqtD6BBiKjaOgTAIKKIxND6y53GnOGWuNSaQlOeCtCywVaEi0jJJaWUphGgNm1QmLS6PYEapq
BcshJQt47iGm3HtWThkczY3a8OYEiS51hqKPnkTsbXM5+RRNgjiQqCghppSgHW8KilRzCIzDlpnO1glyCgSKoIhL2QnhJg2qNuQs
YuI0m5aFhsm0KsQvjEqlI4GR+VqabrW0VmUuBEuQxRQ1Zhur1rZHg0RYluRGwxqBawQApDyQ2niKLkXBSKg4KYfEAqiJUco0kppE
DUMQNINjS15QnwuPVAkxBHRkwSZCAAQ5iZVIDnepQ3ZgAGWDzjQhoQXUFCjuow9DWQkeAiwLWnr3NtuMapPR9Kg4SgWR5lugJoeC
SEjGtxxoRAIAAAAVBBWIBBWSBEwVggEVABIAAIQC9AMBik4AAIhOAAB3TgAAjE4AAHVOAABpTgAAcU4AAJhOAAB9TgAAeE4AAI9O
AACBTgAAlU4AAJJOAAB+TgAAfE4AAJtOAACCTgAAk04AAIVOAACRTgAAe04AAHROAAB2TgAAjk4AAIlOAABzTgAAeU4AAJZOAABw
TgAAgE4AAIdOAACLTgAAhE4AAHJOAAB/TgAAZ04AAG1OAACGTgAAmk4AAHpOAACQTgAAlE4AAKNOAABrTgAAmU4AAI1OAABsTgAA
b04AAINOAABuTgAAak4AAJdOAACrTgAAs04AALROAABjTgAAZk4AALJOAAC1TgAAoU4AAKROAABoTgAAok4AAGROAAAVABWEBxWO
BywV5AcVEBUGFQYcGAS1TgAAGARjTgAAFgAoBLVOAAAYBGNOAAAREQAAAMID9MEBAwAAAOQHAQd/gIBgQCgYDohEQrFgNAaOxKMA
iUgkkQmlYrk8AA9MI6O5PDYci6Dj0WQ+ABBGAKp8QqKIaEQqOUyI00PzuFQGAITmMhmhIomUBqNyrFiB1scleI08JVjgkohVEJzI
JuFxPBiWBCsimwUqCIdoRrkkOJ8AQmJa0DAeBybzeYAyqNSoNUGNaiBWZpH5oCCmBWqyiaQ8j0ViswFVQBvLY7CIyWIvVMYVYV0u
mshIQ0BtIg2UgKBwLUKDDycRyFhQEUvlQ7F9IizCbJJ5WUa3C65EEGw0CoMCNAKEOAIUoOLZmCyXFOcSQ4VgE0IGVVKMEB5MKaRI
IQgLTQhRIZRMH4fhQ3BsNizTxCUYtGaESSjyetAGow8KUZE4EqbH5KHJDRoWmUnlGJg+B5SFswAdRoMKAgSKMDYzAIXgSX10F0dp
8+l0Ap8ZbJSwmCYmBYJmoVQeDcKj1Sk9MAbJLvHJxACYjAXzETwCFEwgxhsFKpFUJKTJKC6ViiR3giUAHUcihuqlHjFBCCLyOQKF
VGIRmHgqjt8koBAsUgRQKeCwJC6MzwXoMCkWA0FA4DGhCDEXyhUDYFQzCgAAAAAAFQQZPDUAGAZzY2hlbWEVBAAVBCUCGAd1c2Vy
X2lkABUCJQIYCmxvZ2luX2RhdGUlDExsAAAAFuQHGRwZLCYAHBUEGTUABhAZGAd1c2VyX2lkFQIW5AcW3AkWqAgmmgImCBwYCCMA
AAAAAAAAGAgBAAAAAAAAABYAKAgjAAAAAAAAABgIAQAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA5AcAAAAmABwVAhk1
AAYQGRgKbG9naW5fZGF0ZRUCFuQHFpIMFqYMJuQMJrAIHBgEtU4AABgEY04AABYAKAS1TgAAGARjTgAAEREAGSwVBBUAFQIAFQAV
EBUCADwpBhkmAOQHAAAAFu4VFuQHJggWzhQAGRwYDEFSUk9XOnNjaGVtYRiAAi8vLy8vN2dBQUFBUUFBQUFBQUFLQUF3QUJnQUZB
QWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFJQUFBQlFBQUFBQkFBQUFNai8vLzhBQUFFSUVBQUFBQ1FB
QUFBRUFBQUFBQUFBQUFvQUFBQnNiMmRwYmw5a1lYUmxBQUFBQUFZQUNBQUdBQVlBQUFBQUFBQUFFQUFVQUFnQUJnQUhBQXdBQUFB
UUFCQUFBQUFBQUFFQ0VBQUFBQ0FBQUFBRUFBQUFBQUFBQUFjQUFBQjFjMlZ5WDJsa0FBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFB
QUEAGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRksHAAAHAAAAFYCAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "028-longest-login-streak"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

app_logins = spark.read.parquet((_data_dir / "app_logins.parquet").resolve().as_posix())
app_logins.createOrReplaceTempView("app_logins")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'streak_days': '4f111d904f829b8a8a53497557487637a928f40a5d9354b03aacf6d4329a9970',
                         'streak_end': 'b2b4a2935c7e23ac5e39b07c6a9d31043ed369747313081cd8479145a37be54c',
                         'streak_start': '1597501c644624af0f446f923d20c36cefc6d72eb39037052ddbcbbac6fdf3fb',
                         'user_id': '93a7d5af1a025129ab36ec0a4d2ade48ea01843e0c3ffeef4ba166349a7ff439'},
 'columns': [['streak_days', 'bigint'],
             ['streak_end', 'date'],
             ['streak_start', 'date'],
             ['user_id', 'bigint']],
 'fingerprint': '15ec2cc9e78fdf56091174ea19c812f3b7cdb5664a04046c6fee0b9d7d4fe7d1',
 'order_matters': False,
 'row_count': 27}

check = make_check(EXPECTED)

## The situation

The app shows every user a badge for their longest login streak: the most days in a
row on which they opened the app.

The badge was wrong for the most active users. People who logged in several times a
day were shown streaks of one or two days, although they had not missed a day in
weeks.

## Input tables

### `app_logins`

One row per login. A user can log in several times on the same day.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | The user. |
| `login_date` | DATE | The calendar date of the login. |

## Requirement

Return each user's longest streak.

- A streak is a run of consecutive calendar days on each of which the user logged in
  at least once. 31 December followed by 1 January is consecutive.
- Several logins on one day count as one day.
- `streak_days` is the number of days in the streak. A user who logged in on a
  single day has a streak of 1.
- If a user has two or more streaks of the same longest length, return the most
  recent of them.

## Expected output

Exactly one row per user.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | |
| `streak_start` | DATE | First day of the streak. |
| `streak_end` | DATE | Last day of the streak. |
| `streak_days` | BIGINT | Number of days from start to end, both included. |

In [ ]:
app_logins.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `app_logins`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)